# Spark Streaming Cookbook

## Table of Contents
1. [Introduction](#introduction)
2. [Getting Started with Spark Streaming](#getting-started-with-spark-streaming)
3. [Core Concepts](#core-concepts)
4. [Basic Spark Streaming Example](#basic-spark-streaming-example)
5. [Datasets Used](#datasets-used)
6. [Scenarios and Solutions](#scenarios-and-solutions)
7. [Advanced Use Cases](#advanced-use-cases)
8. [Performance Tuning](#performance-tuning)
9. [Error Handling and Fault Tolerance](#error-handling-and-fault-tolerance)
10. [Testing Spark Streaming Jobs](#testing-spark-streaming-jobs)
11. [Common Interview Questions](#common-interview-questions)
12. [Conclusion](#conclusion)


## Introduction

### What is Spark Streaming?

Apache Spark Streaming is an extension of the core Spark API that enables scalable, high-throughput, fault-tolerant stream processing of live data streams. It processes live data from various sources (Kafka, Flume, TCP sockets, or local files) in mini-batches using Spark's core RDD (Resilient Distributed Dataset) API.

> **Note**: This handbook focuses on **DStreams** (the original Spark Streaming API). For new projects, consider **Structured Streaming** which offers better performance and ease of use.

### When to Use Spark Streaming (DStreams)

Choose Spark Streaming DStreams when:

- **Legacy Integration**: Working with existing RDD-based Spark applications or legacy codebases
- **RDD Control**: Need full control over RDD transformations and comfortable managing Spark internals
- **Code Reuse**: Want to reuse existing batch logic written in RDD format for streaming contexts
- **Simple Requirements**: Application is straightforward and ultra-low latency is not critical
- **Migration Path**: Gradual migration from batch to streaming processing

This cookbook serves as a practical guide for data engineers and professionals seeking to implement Spark Streaming effectively in real-world scenarios.


## Core Concepts

Understanding these fundamental concepts is essential for working with Spark Streaming:

### DStream (Discretized Stream)
- **Definition**: The basic abstraction in Spark Streaming representing a continuous stream of data
- **Structure**: A sequence of RDDs, where each RDD contains data from a specific time interval
- **Fault Tolerance**: Inherits RDD's fault-tolerance capabilities through lineage tracking

### Batch Interval
- **Purpose**: Time duration defining how streaming data is divided into batches
- **Impact**: Affects latency and throughput trade-offs
- **Typical Range**: 0.5 to 10 seconds depending on use case

### Key Operations
- **Transformations**: Applied to each batch (similar to RDD transformations)
- **Windowed Operations**: Enable aggregations across sliding time windows
- **Output Operations**: Trigger computation and save results (e.g., `print()`, `saveAsTextFiles()`)

### Checkpointing
- **Purpose**: Handles fault recovery and maintains state across application restarts
- **Types**:
  - **Metadata Checkpointing**: Recovery of driver program
  - **Data Checkpointing**: Recovery of generated RDDs

### Data Sources Integration
Spark Streaming supports various input sources:
- **Message Queues**: Kafka, Kinesis, Flume
- **File Systems**: HDFS, S3, local files
- **Network**: TCP sockets, HTTP endpoints
- **Custom Sources**: Through receiver-based or direct approaches

## Basic Spark Streaming Example

In [ ]:
## Environment setup

# Essential imports for basic streaming
from pyspark.sql import SparkSession
from pyspark.streaming import StreamingContext

In [ ]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("BasicStreamingApp") \
    .getOrCreate()

In [ ]:
# Create StreamingContext with 5-second batch intervals
ssc = StreamingContext(spark.sparkContext, 5)

In [ ]:
# Create a DStream that connects to a TCP source at localhost on port 9999
lines = ssc.socketTextStream("localhost", 9999)

In [ ]:
# Transform the data: split lines into words, count word occurrences
words = lines.flatMap(lambda line: line.split(" "))
wordCounts = words.map(lambda word: (word, 1)).reduceByKey(lambda a, b: a + b)

In [ ]:
# Print the result
wordCounts.pprint()

In [ ]:
# Start streaming computation
ssc.start()
ssc.awaitTermination()

How to Simulate Streaming Data (For Testing)

Step 1: *Launch a Terminal to Send Data*

In [ ]:
# On Mac/Linux:
nc -lk 9999

In [ ]:
# On Windows (Requires ncat from Nmap):
ncat -l -p 9999 --keep-open

Step 2: *Type Sample Input*

In [ ]:
# Type one line at a time into the terminal window you just opened:

hello world
spark is awesome
hello spark

In [ ]:
# Expected Output from Spark Streaming - Spark will output aggregated word counts like:

('hello', 2)
('world', 1)
('spark', 2)
('is', 1)
('awesome', 1)

Spark receives each line as it is typed, processes it in batches (every 5 seconds), and prints results to the console.

## Datasets Used

# These are **example records** from streaming data sources used in the scenarios below.
# Used in: Scenario 1 – Real-time Log Level Aggregation

### 1. `logs.csv` - Streaming log data
Represents real-time logs coming from a system.

```csv
timestamp,level,message
2025-08-06 10:00:00,INFO,Service started
2025-08-06 10:00:01,WARN,High memory usage
2025-08-06 10:00:02,ERROR,NullPointerException


# Used in: Scenario 2 - Join Stream with Static Dataset

### 2. `clicks.csv` - User clickstream data
Represents user clicks on a website.

```csv
user_id,timestamp,page_id,action
u1,2025-08-06 10:00:01,home,click
u1,2025-08-06 10:00:05,product_3,click
u2,2025-08-06 10:00:06,home,scroll

## Scenarios and Solutions

In this next sections, we explore more complex transformations and real-world streaming use cases.bold text

In [ ]:
### Scenario 1: Real-time Log Level Aggregation

# Problem: You want to count log levels in real-time.

# Solution:

# Create a DStream that connects to a TCP source (localhost:9998)
logs = ssc.socketTextStream("localhost", 9998)

# Extract the log level (e.g., INFO, WARN, ERROR) from each line
log_levels = logs.map(lambda line: line.split(",")[1])

# Count occurrences of each log level
counts = log_levels.map(lambda level: (level, 1)).reduceByKey(lambda a, b: a + b)

# Print the counts to the console every batch interval
counts.pprint()

# Expected Output

('INFO', 5)
('WARN', 2)
('ERROR', 1)

In [ ]:
### Scenario 2: Join Stream with Static Dataset

# Problem: Match user clickstream with user info.

# Solution:

# Load the dataset containing user info
user_info = spark.read.csv("user_info.csv", header=True)

# Create a DStream for incoming clickstream data
clicks = ssc.socketTextStream("localhost", 9997)

# Define a function to enrich each RDD with static user info
def enrich_clicks(rdd):
    if not rdd.isEmpty():
        # Convert RDD (JSON lines) to DataFrame
        df = spark.read.json(rdd)

        # Join stream data with static user info on 'user_id'
        enriched = df.join(user_info, on="user_id", how="inner")

        # Show enriched result
        enriched.show()

# Apply the enrichment function to each incoming RDD
clicks.foreachRDD(enrich_clicks)



## Performance Tuning


- Use checkpointing (`ssc.checkpoint("hdfs:///checkpoints")`)
- Reduce GC pauses (tune memory)
- Use Kafka instead of sockets for production-grade ingestion
- Batch tuning: 2-5 seconds usually optimal
- Avoid wide transformations like `groupByKey`

## Error Handling and Fault Tolerance

- Use checkpointing for DStreams
- Wrap transformations with try-except blocks
- Monitor job failures using Spark UI

## Testing Spark Streaming Jobs

In [ ]:
from pyspark.streaming import StreamingContext
from collections import deque

# Create a queue of RDDs to simulate input stream data
rdd_queue = deque([
    ssc.sparkContext.parallelize(["a b c", "b c d"])  # Each string is a record
])

# Create a DStream from the queued RDDs
input_stream = ssc.queueStream(rdd_queue)


## Common Interview Questions

1. Explain how Spark Streaming handles backpressure.
*Answer*: Spark Streaming handles backpressure by dynamically adjusting the rate at which data is ingested from sources like Kafka. When spark.streaming.backpressure.enabled is set to true, Spark monitors processing time and memory usage, then reduces the ingestion rate if batches are not being processed fast enough. This prevents the system from being overwhelmed by incoming data.

2. What is the difference between DStream and Structured Streaming?
*Answer*: DStream (Discretized Stream) is the original API, based on RDDs and micro-batches. It has less optimization, less fault-tolerant support for complex operations, and limited integration with structured data.

Structured Streaming is a newer, higher-level API built on DataFrames and Catalyst optimizer. It allows declarative queries, better fault tolerance, supports event time and watermarking, and integrates with batch and streaming seamlessly.

3. How do you ensure fault tolerance in Spark Streaming?
*Answer*: Use checkpointing with ssc.checkpoint() to persist metadata and state.

For input sources like Kafka, offsets can be tracked and committed manually.

Spark automatically retries failed tasks and restores lost RDDs using lineage.

Wrap critical operations in try-except blocks to handle data-specific errors.

4. How can you manage late data in streaming?
*Answer*: In Structured Streaming, use watermarking (e.g., withWatermark()) to handle late events by defining how long to wait before considering data "too late."
In DStreams, late data is harder to manage but can be approximated using windowed operations with longer durations, though with less precision.

5. When would you prefer Kafka over sockets?
Kafka is preferred for:
*Answer*: Production use cases (scalable, fault-tolerant, durable)

Exactly-once or at-least-once semantics

Multiple consumers or replayable streams

Backpressure support and load balancing

Sockets (socketTextStream) are only suitable for local testing, as they do not scale or persist data.



## Conclusion

This cookbook provides real-world usage examples of Spark Streaming for data engineers, offering hands-on scenarios, tuning tips, and guidance for interviews. For further exploration, consider transitioning to Structured Streaming for modern scalable applications.